In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Loading 

In [ ]:
import pandas as pd
import os

# Define the base directory for the competition data
# In a Kaggle environment, this path is typical.
base_path = "/kaggle/input/house-prices-advanced-regression-techniques"

# --- 1. Load the datasets ---

# Path to the training data
train_file_path = os.path.join(base_path, 'train.csv')
print(f"Attempting to load training data from: {train_file_path}")

try:
    # Load train.csv into a DataFrame named train_df
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully!")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct in your environment.")
    # Create an empty DataFrame as a fallback in case the environment isn't Kaggle
    train_df = pd.DataFrame() 

# Path to the testing data
test_file_path = os.path.join(base_path, 'test.csv')
print(f"Attempting to load testing data from: {test_file_path}")

try:
    # Load test.csv into a DataFrame named test_df
    test_df = pd.read_csv(test_file_path)
    print("Testing data loaded successfully!")
except FileNotFoundError:
    print(f"Error: Testing file not found at {test_file_path}. Please ensure the path is correct in your environment.")
    # Create an empty DataFrame as a fallback
    test_df = pd.DataFrame()

# --- 2. Display the first 5 rows of train_df ---

if not train_df.empty:
    print("\n--- First 5 rows of train_df (Training Dataset Head) ---")
    print(train_df.head())
    print(f"\ntrain_df shape: {train_df.shape}")
else:
    print("\nCannot display train_df head as the DataFrame is empty due to file loading failure.")

if not test_df.empty:
    print(f"test_df shape: {test_df.shape}")

In [ ]:
# --- 3. Exploratory Data Analysis (EDA) on train_df ---
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns

if not train_df.empty:
    print("\n" + "="*60)
    print("--- 3. Exploratory Data Analysis (EDA) on train_df ---")
    print("="*60)

    # 3.1 Basic Structure and Statistics
    print("\n[3.1] Data Structure (train_df.info()):")
    # This includes column names, non-null counts, and data types
    train_df.info()

    print("\n[3.2] Descriptive Statistics (train_df.describe()):")
    # This provides count, mean, std, min, max, and quartiles for numeric columns
    print(train_df.describe())

    # 3.3 Histogram of SalePrice (Raw and Log Transformed)
    print("\n[3.3] Generating SalePrice Distribution Plots...")
    plt.figure(figsize=(14, 6))

    # Raw SalePrice Distribution: shows skewness
    plt.subplot(1, 2, 1)
    sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='teal')
    plt.title('Raw SalePrice Distribution (Target Variable)')
    plt.xlabel('SalePrice ($)')
    plt.ylabel('Frequency')

    # Log-Transformed SalePrice Distribution: shows normalized distribution
    plt.subplot(1, 2, 2)
    # np.log1p computes log(1+x), which is useful for transforming positive skewed data
    log_sale_price = np.log1p(train_df['SalePrice'])
    sns.histplot(log_sale_price, kde=True, bins=50, color='purple')
    plt.title('Log-Transformed SalePrice Distribution')
    plt.xlabel('Log(1 + SalePrice)')
    plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

    # 3.4 Correlation Heatmap of Top 10 Features with SalePrice
    print("\n[3.4] Generating Correlation Heatmap of Top 10 Features...")
    
    # Select only numeric columns for correlation
    numeric_cols = train_df.select_dtypes(include=np.number).columns
    corr_matrix = train_df[numeric_cols].corr()
    
    # Get the correlation with SalePrice and sort it
    sale_price_corr = corr_matrix['SalePrice'].sort_values(ascending=False)
    
    # Select the top 10 features (excluding SalePrice itself, then adding it back for the matrix)
    # This ensures the 10 strongest correlating features are plotted against each other and SalePrice
    top_10_features_names = sale_price_corr[1:11].index.tolist()
    features_to_plot = ['SalePrice'] + top_10_features_names

    # Calculate the correlation matrix for only these top features
    top_corr_matrix = train_df[features_to_plot].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(top_corr_matrix, 
                annot=True,        # Display correlation values on the heatmap
                fmt=".2f",         # Format numbers to two decimal places
                cmap='coolwarm',   # Color map for better visual distinction
                linewidths=.5,     # Lines between cells
                linecolor='black',
                cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap of Top 10 Numeric Features vs SalePrice')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3.5 Boxplot of Categorical Variable (Neighborhood vs SalePrice)
    print("\n[3.5] Generating Boxplot for Neighborhood vs SalePrice...")
    plt.figure(figsize=(18, 7))
    
    # Order the neighborhoods by their median SalePrice for easier interpretation
    sorted_nbh = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index
    
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, order=sorted_nbh, palette='Set2')
    
    plt.title('SalePrice Distribution by Neighborhood')
    plt.xlabel('Neighborhood (Sorted by Median SalePrice)')
    plt.ylabel('SalePrice ($)')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    print("\nEDA section complete. Plots have been generated and displayed.")
else:
    print("\nSkipping EDA section as train_df is empty due to file loading failure.")

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Note: This code assumes 'train_df' has been successfully loaded 
# from the initial steps in your notebook.

print("--- Data Splitting (80/20) ---")

# 1. Define features (X) and target (y)
# X includes all columns except 'SalePrice'
X = train_df.drop('SalePrice', axis=1)
# y is the target variable 'SalePrice'
y = train_df['SalePrice']

# 2. Split the data into 80% training and 20% validation/testing sets
# random_state=42 is used for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Original features shape: {X.shape}")
print(f"Original target shape: {y.shape}")

print("\n--- Data Split Shapes ---")
print(f"X_train (Training Features): {X_train.shape}")
print(f"X_test (Validation Features): {X_test.shape}")
print(f"y_train (Training Target): {y_train.shape}")
print(f"y_test (Validation Target): {y_test.shape}")

print("\nData splitting complete.")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# --- 5. Train a Linear Regression Model with Preprocessing Pipeline ---

print("--- 5. Training Linear Regression Model ---")

# Note: We must handle both categorical and numerical features 
# before training the Linear Regression model.

# 5.1 Identify column types
# Drop 'Id' if it was kept in X_train (often unnecessary for training)
if 'Id' in X_train.columns:
    X_train = X_train.drop('Id', axis=1)

numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print(f"Numerical Features Count: {len(numerical_features)}")
print(f"Categorical Features Count: {len(categorical_features)}")
print("-" * 50)


# 5.2 Create Preprocessing Steps (Pipelines)

# Preprocessing for Numerical Data: Impute Median, then Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for Categorical Data: Impute 'missing', then One-Hot Encode
# We use handle_unknown='ignore' for safety when transforming test data later
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


# 5.3 Combine Preprocessors using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep any other columns (shouldn't be any here)
)

# 5.4 Create the Full Model Pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 5.5 Train the Model
print("Fitting the full model pipeline to X_train and y_train...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")


# 5.6 Evaluate on Training Data (Quick Check)
y_train_pred = model_pipeline.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = model_pipeline.score(X_train, y_train)

print("-" * 50)
print("--- Training Set Performance ---")
print(f"R-squared (R2 Score): {r2_train:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_train:,.2f}")
print("Note: The model score is based on the raw (non-log) SalePrice.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# --- 5. Train a Linear Regression Model with Preprocessing Pipeline ---

print("--- 5. Training Linear Regression Model ---")

# Note: We must handle both categorical and numerical features 
# before training the Linear Regression model.

# 5.1 Identify column types
# Drop 'Id' if it was kept in X_train (often unnecessary for training)
if 'Id' in X_train.columns:
    X_train = X_train.drop('Id', axis=1)

numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print(f"Numerical Features Count: {len(numerical_features)}")
print(f"Categorical Features Count: {len(categorical_features)}")
print("-" * 50)


# 5.2 Create Preprocessing Steps (Pipelines)

# Preprocessing for Numerical Data: Impute Median, then Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for Categorical Data: Impute 'missing', then One-Hot Encode
# We use handle_unknown='ignore' for safety when transforming test data later
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


# 5.3 Combine Preprocessors using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep any other columns (shouldn't be any here)
)

# 5.4 Create the Full Model Pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 5.5 Train the Model
print("Fitting the full model pipeline to X_train and y_train...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")


# 5.6 Evaluate on Training Data (Quick Check)
y_train_pred = model_pipeline.predict(X_train)
mse_train = mean_squared_error(y_train, y_train_pred) # Explicitly calculate MSE
rmse_train = np.sqrt(mse_train) # Calculate RMSE
r2_train = model_pipeline.score(X_train, y_train)

print("-" * 50)
print("--- Training Set Performance ---")
print(f"R-squared (R2 Score): {r2_train:.4f}")
print(f"Mean Squared Error (MSE): ${mse_train:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_train:,.2f}")


# 5.7 Evaluate on Validation Data (X_test, y_test) - Added for Overfitting Check
y_test_pred = model_pipeline.predict(X_test)
mse_test = mean_squared_error(y_test, y_test_pred) # Explicitly calculate MSE
rmse_test = np.sqrt(mse_test) # Calculate RMSE
r2_test = model_pipeline.score(X_test, y_test)

print("-" * 50)
print("--- Validation Set Performance ---")
print(f"R-squared (R2 Score): {r2_test:.4f}")
print(f"Mean Squared Error (MSE): ${mse_test:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_test:,.2f}")
print("Note: The model score is based on the raw (non-log) SalePrice.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # Corrected: Removed extra '_linear'
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
# Import necessary base classes for custom transformer
from sklearn.base import BaseEstimator, TransformerMixin 


# --- Custom Feature Engineering Transformer ---
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    A custom transformer to implement feature engineering steps like 
    creating Total SF and Total Bathrooms features.
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure a copy is made to avoid modifying the original DataFrame
        X_copy = X.copy()
        
        # 1. Total Square Footage
        # Note: We must impute NaNs with 0 before summing to avoid NaN result
        X_copy['TotalBsmtSF'] = X_copy['TotalBsmtSF'].fillna(0)
        X_copy['1stFlrSF'] = X_copy['1stFlrSF'].fillna(0)
        X_copy['2ndFlrSF'] = X_copy['2ndFlrSF'].fillna(0)
        
        X_copy['TotalSF'] = (
            X_copy['TotalBsmtSF'] + 
            X_copy['1stFlrSF'] + 
            X_copy['2ndFlrSF']
        )
        
        # 2. Total Bathrooms
        # Note: Must impute NaNs with 0 before summing
        X_copy['BsmtFullBath'] = X_copy['BsmtFullBath'].fillna(0)
        X_copy['BsmtHalfBath'] = X_copy['BsmtHalfBath'].fillna(0)
        X_copy['FullBath'] = X_copy['FullBath'].fillna(0)
        X_copy['HalfBath'] = X_copy['HalfBath'].fillna(0)

        X_copy['TotalBath'] = (
            X_copy['BsmtFullBath'] + 
            (0.5 * X_copy['BsmtHalfBath']) + 
            X_copy['FullBath'] + 
            (0.5 * X_copy['HalfBath'])
        )
        
        # 3. Age of House and Age since Remodel
        X_copy['AgeAtSale'] = X_copy['YrSold'] - X_copy['YearBuilt']
        X_copy['YearsSinceRemod'] = X_copy['YrSold'] - X_copy['YearRemodAdd']
        
        # Handle cases where remod year is before built year (shouldn't happen 
        # but prevents negative ages if data is messy)
        X_copy['YearsSinceRemod'] = X_copy['YearsSinceRemod'].apply(lambda x: x if x >= 0 else 0)
        X_copy['AgeAtSale'] = X_copy['AgeAtSale'].apply(lambda x: x if x >= 0 else 0)


        # Remove redundant original columns (optional but good practice)
        X_copy = X_copy.drop(columns=['TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                                      'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 
                                      'YrSold', 'YearBuilt', 'YearRemodAdd'], 
                             errors='ignore')
        
        return X_copy
# ----------------------------------------------------------------------


# --- 5. Train a Linear Regression Model with Preprocessing Pipeline ---

print("--- 5. Training Linear Regression Model ---")
print("\n--- NEW: Feature Engineering Implemented (TotalSF, TotalBath, Age) ---") # Added print statement

# Note: We must handle both categorical and numerical features 
# before training the Linear Regression model.

# 5.1 Identify column types (Determined AFTER initial features are dropped/created by the FeatureEngineer)
# We must first apply the transformer to X_train temporarily to get the new column names
# The full pipeline will handle the application properly, but we need the final feature list for the ColumnTransformer setup.
fe_transformer = FeatureEngineer()
X_train_transformed = fe_transformer.fit_transform(X_train) # Use the fit_transform method

# Drop 'Id' if it was kept in X_train_transformed
if 'Id' in X_train_transformed.columns:
    X_train_transformed = X_train_transformed.drop('Id', axis=1)

numerical_features = X_train_transformed.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train_transformed.select_dtypes(include='object').columns.tolist()

print(f"Numerical Features Count (New): {len(numerical_features)}")
print(f"Categorical Features Count (New): {len(categorical_features)}")
print("-" * 50)


# 5.2 Create Preprocessing Steps (Pipelines) - No change needed here


# Preprocessing for Numerical Data: Impute Median, then Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for Categorical Data: Impute 'missing', then One-Hot Encode
# We use handle_unknown='ignore' for safety when transforming test data later
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


# 5.3 Combine Preprocessors using ColumnTransformer
# IMPORTANT: The ColumnTransformer now uses the updated feature lists
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep any other columns (shouldn't be any here)
)

# 5.4 Create the Full Model Pipeline
# NEW: Insert the FeatureEngineer as the first step
model_pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()), # Step 1: Feature Engineering
    ('preprocessor', preprocessor),          # Step 2: Impute, Scale, One-Hot Encode
    ('regressor', LinearRegression())        # Step 3: Model Training
])

# 5.5 Train the Model
print("Fitting the full model pipeline (Feature Engineering included) to X_train and y_train...")
model_pipeline.fit(X_train, y_train) # Fit the pipeline to the ORIGINAL X_train
print("Model training complete.")


# 5.6 Evaluate on Training Data (Quick Check)
y_train_pred = model_pipeline.predict(X_train)
mse_train = mean_squared_error(y_train, y_train_pred) # Explicitly calculate MSE
rmse_train = np.sqrt(mse_train) # Calculate RMSE
r2_train = model_pipeline.score(X_train, y_train)

print("-" * 50)
print("--- Training Set Performance (New Features) ---")
print(f"R-squared (R2 Score): {r2_train:.4f}")
print(f"Mean Squared Error (MSE): ${mse_train:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_train:,.2f}")


# 5.7 Evaluate on Validation Data (X_test, y_test) - Added for Overfitting Check
y_test_pred = model_pipeline.predict(X_test)
mse_test = mean_squared_error(y_test, y_test_pred) # Explicitly calculate MSE
rmse_test = np.sqrt(mse_test) # Calculate RMSE
r2_test = model_pipeline.score(X_test, y_test)

print("-" * 50)
print("--- Validation Set Performance (New Features) ---")
print(f"R-squared (R2 Score): {r2_test:.4f}")
print(f"Mean Squared Error (MSE): ${mse_test:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_test:,.2f}")
print("Note: The model score is based on the raw (non-log) SalePrice.")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
# Import necessary base classes for custom transformer
from sklearn.base import BaseEstimator, TransformerMixin 
# NEW IMPORTS for advanced models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor


# --- Custom Feature Engineering Transformer ---
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    A custom transformer to implement feature engineering steps like 
    creating Total SF and Total Bathrooms features.
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure a copy is made to avoid modifying the original DataFrame
        X_copy = X.copy()
        
        # 1. Total Square Footage
        # Note: We must impute NaNs with 0 before summing to avoid NaN result
        X_copy['TotalBsmtSF'] = X_copy['TotalBsmtSF'].fillna(0)
        X_copy['1stFlrSF'] = X_copy['1stFlrSF'].fillna(0)
        X_copy['2ndFlrSF'] = X_copy['2ndFlrSF'].fillna(0)
        
        X_copy['TotalSF'] = (
            X_copy['TotalBsmtSF'] + 
            X_copy['1stFlrSF'] + 
            X_copy['2ndFlrSF']
        )
        
        # 2. Total Bathrooms
        # Note: Must impute NaNs with 0 before summing
        X_copy['BsmtFullBath'] = X_copy['BsmtFullBath'].fillna(0)
        X_copy['BsmtHalfBath'] = X_copy['BsmtHalfBath'].fillna(0)
        X_copy['FullBath'] = X_copy['FullBath'].fillna(0)
        X_copy['HalfBath'] = X_copy['HalfBath'].fillna(0)

        X_copy['TotalBath'] = (
            X_copy['BsmtFullBath'] + 
            (0.5 * X_copy['BsmtHalfBath']) + 
            X_copy['FullBath'] + 
            (0.5 * X_copy['HalfBath'])
        )
        
        # 3. Age of House and Age since Remodel
        X_copy['AgeAtSale'] = X_copy['YrSold'] - X_copy['YearBuilt']
        X_copy['YearsSinceRemod'] = X_copy['YrSold'] - X_copy['YearRemodAdd']
        
        # Handle cases where remod year is before built year (shouldn't happen 
        # but prevents negative ages if data is messy)
        X_copy['YearsSinceRemod'] = X_copy['YearsSinceRemod'].apply(lambda x: x if x >= 0 else 0)
        X_copy['AgeAtSale'] = X_copy['AgeAtSale'].apply(lambda x: x if x >= 0 else 0)


        # Remove redundant original columns (optional but good practice)
        X_copy = X_copy.drop(columns=['TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                                      'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 
                                      'YrSold', 'YearBuilt', 'YearRemodAdd'], 
                             errors='ignore')
        
        return X_copy
# ----------------------------------------------------------------------


# --- 5. Train a Linear Regression Model with Preprocessing Pipeline ---

print("--- 5. Training Linear Regression Model ---")
print("\n--- NEW: Feature Engineering Implemented (TotalSF, TotalBath, Age) ---") # Added print statement

# Note: We must handle both categorical and numerical features 
# before training the Linear Regression model.

# 5.1 Identify column types (Determined AFTER initial features are dropped/created by the FeatureEngineer)
# We must first apply the transformer to X_train temporarily to get the new column names
# The full pipeline will handle the application properly, but we need the final feature list for the ColumnTransformer setup.
fe_transformer = FeatureEngineer()
# Note: X_train is assumed to be defined from a previous step (data loading/split)
X_train_transformed = fe_transformer.fit_transform(X_train) # Use the fit_transform method

# Drop 'Id' if it was kept in X_train_transformed
if 'Id' in X_train_transformed.columns:
    X_train_transformed = X_train_transformed.drop('Id', axis=1)

numerical_features = X_train_transformed.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train_transformed.select_dtypes(include='object').columns.tolist()

print(f"Numerical Features Count (New): {len(numerical_features)}")
print(f"Categorical Features Count (New): {len(categorical_features)}")
print("-" * 50)


# 5.2 Create Preprocessing Steps (Pipelines) - No change needed here


# Preprocessing for Numerical Data: Impute Median, then Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for Categorical Data: Impute 'missing', then One-Hot Encode
# We use handle_unknown='ignore' for safety when transforming test data later
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


# 5.3 Combine Preprocessors using ColumnTransformer
# IMPORTANT: The ColumnTransformer now uses the updated feature lists
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep any other columns (shouldn't be any here)
)

# 5.4 Create the Full Model Pipeline
# NEW: Insert the FeatureEngineer as the first step
model_pipeline_lr = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()), # Step 1: Feature Engineering
    ('preprocessor', preprocessor),          # Step 2: Impute, Scale, One-Hot Encode
    ('regressor', LinearRegression())        # Step 3: Model Training
])

# 5.5 Train the Model
print("Fitting the Linear Regression pipeline...")
model_pipeline_lr.fit(X_train, y_train) # Fit the pipeline to the ORIGINAL X_train
print("Linear Regression model training complete.")


# 5.6 Evaluate on Training Data (Quick Check)
y_train_pred_lr = model_pipeline_lr.predict(X_train)
mse_train_lr = mean_squared_error(y_train, y_train_pred_lr) # Explicitly calculate MSE
rmse_train_lr = np.sqrt(mse_train_lr) # Calculate RMSE
r2_train_lr = model_pipeline_lr.score(X_train, y_train)

print("-" * 50)
print("--- LR Training Set Performance (New Features) ---")
print(f"R-squared (R2 Score): {r2_train_lr:.4f}")
print(f"Mean Squared Error (MSE): ${mse_train_lr:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_train_lr:,.2f}")


# 5.7 Evaluate on Validation Data (X_test, y_test) - Added for Overfitting Check
y_test_pred_lr = model_pipeline_lr.predict(X_test)
mse_test_lr = mean_squared_error(y_test, y_test_pred_lr) # Explicitly calculate MSE
rmse_test_lr = np.sqrt(mse_test_lr) # Calculate RMSE
r2_test_lr = model_pipeline_lr.score(X_test, y_test)

print("-" * 50)
print("--- LR Validation Set Performance (New Features) ---")
print(f"R-squared (R2 Score): {r2_test_lr:.4f}")
print(f"Mean Squared Error (MSE): ${mse_test_lr:,.2f}") # Display MSE
print(f"Root Mean Squared Error (RMSE): ${rmse_test_lr:,.2f}")
print("Note: The LR model score is based on the raw (non-log) SalePrice.")


# --- 6. Train and Compare Advanced Models ---
print("\n" + "="*60)
print("--- 6. Training and Comparing Tree-Based Models ---")
print("="*60)

# Define models to compare
models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, max_depth=5)
}

# Dictionary to store results
results = {'LinearRegression': {'RMSE': rmse_test_lr}}

# Loop through models, train, and evaluate
for name, estimator in models.items():
    print(f"\nTraining {name} Regressor...")
    
    # Create a new pipeline for the current model, reusing FE and preprocessor
    model_pipeline = Pipeline(steps=[
        ('feature_engineer', FeatureEngineer()), 
        ('preprocessor', preprocessor), 
        ('regressor', estimator)
    ])
    
    # Train
    model_pipeline.fit(X_train, y_train)
    
    # Predict on Validation Set
    y_test_pred = model_pipeline.predict(X_test)
    
    # Calculate Metrics
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    # Store result
    results[name] = {'RMSE': rmse_test}
    print(f"{name} training complete. Validation RMSE: ${rmse_test:,.2f}")

# Display Comparison Summary
print("\n" + "="*60)
print("--- Model Comparison Summary (Validation RMSE) ---")
print("="*60)

# Sort results by RMSE (lower is better)
sorted_results = dict(sorted(results.items(), key=lambda item: item[1]['RMSE']))

for name, metrics in sorted_results.items():
    print(f"{name:<20} | RMSE: ${metrics['RMSE']:,.2f}")
    
print("-" * 60)
print("The model with the lowest validation RMSE is generally considered the best.")

In [ ]:
print("\n" + "="*60)
print("--- 7. Final Model Training, Prediction, and Submission ---")
print("="*60)

# The best model is typically XGBoost (or the lowest RMSE from Section 6). 
# We will use the XGBoost configuration for the final submission.

try:
    # 7.1 Define and Retrain the final XGBoost model on the full training dataset (X and y)
    # We must assume 'train_df' and 'test_df' are available from the initial loading step.
    X_full = train_df.drop('SalePrice', axis=1, errors='ignore') 
    y_full = train_df['SalePrice']
    
    # Use the best model configuration (XGBoost)
    final_xgb_estimator = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, max_depth=5)

    # Create the final XGBoost pipeline structure
    final_xgb_pipeline = Pipeline(steps=[
        ('feature_engineer', FeatureEngineer()), 
        ('preprocessor', preprocessor), 
        ('regressor', final_xgb_estimator)
    ])

    print("Retraining the FINAL XGBoost model on the full training set (X_full, y_full)...")
    final_xgb_pipeline.fit(X_full, y_full)
    print("Final model training complete.")

    # 7.2 Generate predictions on the test_df
    # Assuming test_df is available from the data loading step
    print(f"Generating predictions on test_df (Shape: {test_df.shape})...")
    test_predictions = final_xgb_pipeline.predict(test_df)
    
    # 7.3 Create submission file
    submission_df = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': test_predictions})
    submission_file_path = 'submission.csv'
    submission_df.to_csv(submission_file_path, index=False)

    print("-" * 60)
    print(f"Submission file created successfully!")
    print(f"File: {submission_file_path}")
    print(f"First 5 predictions:\n{submission_df.head()}")
    
except NameError as e:
    print("-" * 60)
    print(f"ERROR: Cannot perform final prediction/submission. A required DataFrame (train_df or test_df) is missing.")
    print(f"Please ensure the data loading steps defining 'train_df' and 'test_df' were executed first.")
    print(f"Original Error: {e}")